# Hotel Booking Dataset Cleaning

Clean and validate the messy hotel booking dataset.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

input_path = "Hotel_Messy_41.csv"

df = pd.read_csv(input_path)
print("Loaded shape:", df.shape)

Loaded shape: (41, 8)


## Standardize Text and Missing Values

In [16]:
str_cols = [
    col for col in df.columns
    if pd.api.types.is_string_dtype(df[col])
]
for col in str_cols:
    df[col] = df[col].astype("string").str.strip()

missing_tokens = {"", "na", "n/a", "null", "none", "unknown", "not available", "-"}
for col in str_cols:
    df[col] = df[col].mask(df[col].str.lower().isin(missing_tokens))

print("Missing values after standardization:")
print(df.isna().sum())

Missing values after standardization:
BookingID        0
GuestID          0
Gender           0
Age              2
RoomType         0
RoomRate         1
BookingStatus    0
City             3
dtype: int64


## Normalize Categorical Columns

In [ ]:
df["Gender"] = (
    df["Gender"]
    .str.upper()
    .map({"F": "Female", "FEMALE": "Female", "M": "Male", "MALE": "Male"})
)

print("Gender values:", sorted(df["Gender"].dropna().unique()))

Gender values: ['Female', 'Male']
Room types: ['Deluxe', 'Executive', 'Standard', 'Suite']
Booking statuses: ['Cancelled', 'Checked Out', 'Confirmed']
Cities: ['Bangalore', 'Chennai', 'Delhi', 'Goa', 'Mumbai', 'Pune']


In [ ]:
df["BookingStatus"] = df["BookingStatus"].str.title()

print("Booking statuses:", sorted(df["BookingStatus"].dropna().unique()))

In [ ]:
df["RoomType"] = df["RoomType"].str.title()

print("Room types:", sorted(df["RoomType"].dropna().unique()))

In [ ]:
df["City"] = df["City"].str.title()

print("Cities:", sorted(df["City"].dropna().unique()))

## Clean Numeric Columns and Fill Missing Values

In [ ]:
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df.loc[~df["Age"].between(18, 100), "Age"] = np.nan
df["Age"] = df["Age"].fillna(df["Age"].median()).round().astype("Int64")

print(df["Age"].describe())

             Age     RoomRate
count       41.0         41.0
mean   40.829268  4204.878049
std    13.903062  1465.426767
min         20.0       2300.0
25%         32.0       2900.0
50%         38.0       4400.0
75%         50.0       5500.0
max         70.0       7000.0


In [ ]:
df["RoomRate"] = (
    df["RoomRate"]
    .astype("string")
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors="coerce")
)
df.loc[df["RoomRate"] < 0, "RoomRate"] = np.nan
df["RoomRate"] = df["RoomRate"].fillna(df["RoomRate"].median())

print(df["RoomRate"].describe())

## Remove Duplicate Bookings and Validate

In [19]:
duplicate_count = df["BookingID"].duplicated().sum()
df = df.drop_duplicates(subset="BookingID", keep="first").reset_index(drop=True)

assert df["BookingID"].is_unique
assert df["Gender"].dropna().isin(["Female", "Male"]).all()
assert df["RoomType"].dropna().isin(["Standard", "Deluxe", "Executive", "Suite"]).all()
assert df["BookingStatus"].dropna().isin(["Confirmed", "Cancelled", "Checked Out"]).all()
assert df["Age"].between(18, 100).all()
assert df["RoomRate"].ge(0).all()

print("Duplicates removed:", duplicate_count)
print("Final missing values:")
print(df.isna().sum())

Duplicates removed: 1
Final missing values:
BookingID        0
GuestID          0
Gender           0
Age              0
RoomType         0
RoomRate         0
BookingStatus    0
City             2
dtype: int64


## Save the Cleaned Dataset

In [ ]:
output_path = "Hotel_Cleaned_41.csv"
df.to_csv(output_path, index=False)

cleaned_df = pd.read_csv(output_path)
print("Saved to:", output_path)
print("Saved shape:", cleaned_df.shape)

Saved to: c:\Users\Dharshan\Desktop\Data_Cleaning\Hotel\Hotel_Cleaned_41.csv
Saved shape: (40, 8)
